In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

TARGET = "distance"
TRAIN_FRACTION = 0.8
SEED = 42

root = Path.cwd()
if not (root / "dataset" / "riemann_features.csv").is_file():
    root = root.parent

results_dir = root / "results" / "NN"
dataset = pd.read_csv(
    root / "dataset" / "riemann_features.csv",
    usecols=lambda column: "lag" not in column.casefold(),
).sort_values("gram", kind="stable", ignore_index=True)

if not np.isfinite(dataset.to_numpy()).all():
    raise ValueError("O CSV contém valores ausentes ou não finitos.")

train_end = int(TRAIN_FRACTION * len(dataset))

In [ ]:
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

X = dataset.iloc[:train_end - 1].drop(columns=TARGET)
y = dataset[TARGET].iloc[1:train_end]
split = int(0.75 * len(X))
X_fit, X_rank = X.iloc[:split], X.iloc[split:]
y_fit, y_rank = y.iloc[:split], y.iloc[split:]

mi = mutual_info_regression(X_fit, y_fit, random_state=SEED, n_neighbors=5)
tree = dict(learning_rate=0.05, subsample=0.8, colsample_bytree=0.7, random_state=SEED, n_jobs=2)
xgb = XGBRegressor(n_estimators=300, max_depth=5, **tree)
xgb.fit(X_fit, y_fit)
permutation = permutation_importance(xgb, X_rank, y_rank, n_repeats=15, random_state=SEED)

rfe = RFE(
    XGBRegressor(n_estimators=100, max_depth=4, **tree),
    n_features_to_select=1,
    step=1,
)
rfe.fit(X_fit, y_fit)

consensus = pd.DataFrame({
    "MI": pd.Series(mi, index=X.columns).rank(ascending=False),
    "Permutation": pd.Series(permutation.importances_mean, index=X.columns).rank(ascending=False),
    "RFE": pd.Series(rfe.ranking_, index=X.columns).rank(),
})
consensus["avg_rank"] = consensus.mean(axis=1)
consensus = consensus.sort_index().sort_values("avg_rank", kind="stable")

results_dir.mkdir(parents=True, exist_ok=True)
consensus.to_csv(results_dir / "feature_ranking_full_nn.csv")
consensus